# URAx-LACE — ALQAC 2026 (one-shot Colab runner)

**Legal Agentic Case-outcome Engine.** Run the cells top-to-bottom.
Recommended runtime: **A100 / L4 / Blackwell GPU** (High-RAM).

Outputs are saved to Google Drive under `ALQAC_RESULT/run_<split>/`. The final file to
upload is `[submission] URAx.json`.

> **Resumable.** If Colab disconnects, just re-run the last cell — cached API evidence
> and checkpoints on Drive are reused, so **no API calls are wasted**.
>
> **Backend.** vLLM is used if its CUDA build matches the runtime; otherwise the pipeline
> automatically falls back to the transformers (HF) backend. Either way it just runs.

## 1. Clone the repository

In [ ]:
import os
if not os.path.exists('ALQAC2026-URAx'):
    !git clone https://github.com/KamonHuiz/ALQAC2026-URAx.git
%cd ALQAC2026-URAx
!git pull -q

## 2. Mount Drive, save the token (once), and drop in the private test file

Token resolution order: `ALQAC_TOKEN` env → Colab Secret `ALQAC_TOKEN` → `ALQAC_RESULT/token.txt`.
Paste your token in `MY_TOKEN` below — it is written to Drive (private to you), never to git.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

ROOT = '/content/drive/MyDrive/ALQAC_RESULT'
os.makedirs(ROOT + '/input', exist_ok=True)   # <-- create folders FIRST

MY_TOKEN = ''  # <-- paste your alqac_... token here (kept on Drive only, never in git)
if MY_TOKEN:
    with open(ROOT + '/token.txt', 'w') as f:
        f.write(MY_TOKEN.strip())
    os.environ['ALQAC_TOKEN'] = MY_TOKEN.strip()
    print('Token saved to', ROOT + '/token.txt')
elif os.path.exists(ROOT + '/token.txt'):
    print('Using existing token at', ROOT + '/token.txt')
else:
    print('WARNING: no token set. Paste it into MY_TOKEN or create', ROOT + '/token.txt')

print('\nPut the 60-case PRIVATE TEST json into:', ROOT + '/input/')
print('(auto-detected: any *.json there whose items have a case_query field)')

## 3. Quick sanity check — offline validation on the labelled public set (no API)

First run installs deps (a few minutes). Measures outcome accuracy + law micro-F1 on the
50 public cases. Great for iterating before spending any API budget.

In [ ]:
!bash scripts/run_colab.sh --split public --no-api --group URAx

## 4. The real run — PRIVATE test (uses the API, ~2-3h due to the 5s rate limit)

Fully resumable: re-run this cell after any disconnect; cached evidence is reused.

In [ ]:
!bash scripts/run_colab.sh --split private --group URAx

## 5. Inspect the submission

In [ ]:
import json, glob
subs = sorted(glob.glob('/content/drive/MyDrive/ALQAC_RESULT/*/submission.json'))
path = subs[-1]
print('Latest submission:', path)
sub = json.load(open(path, encoding='utf-8'))
print(len(sub), 'cases')
print(json.dumps(sub[0], ensure_ascii=False, indent=2))